[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/feat/broad-obesity-2/competitions/broad-obesity-2/quickstarters/perturbed-mean-baseline/perturbed-mean-baseline.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/feat/broad-obesity-2/competitions/broad-obesity-2/assets/banner.webp)

# Obesity ML Competition: Tackling metabolic diseases

## Crunch 2 – Predicting the effect of held-out double-gene perturbations

In Crunch 1, we explored the single-cell transcriptomic response to **single-gene** perturbations that were not provided in the training dataset.

In Crunch 2, we will explore how well we can predict the single-cell transcriptomic response to **double-gene** perturbations that were not provided in the training dataset.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.io/competitions/broad-obesity-2/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [ ]:
# Install the Crunch CLI
# %pip install --upgrade crunch-cli

# Setup your local environment
# !crunch setup-notebook broad-obesity-2 aaaabbbbccccddddeeeeffff

# Your model

## Setup

In [ ]:
# Install required dependencies
%pip install anndata scanpy

In [ ]:
# Necessary for the staging environment
%env CRUNCH_COMPETITIONS_BRANCH=feat/broad-obesity-2
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

In [4]:
import gc
import os
import typing

# Import your dependencies
import anndata
import h5py
import numpy as np
import pandas as pd
import psutil
import scanpy
import scipy
from tqdm.notebook import tqdm
from typing import Dict, List

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Constants

Store your global values here.

In [6]:
# @crunch/keep:on

# Label identifying negative control cells (no gene perturbation), used as the baseline reference
single_control_label = "NC"
double_control_label = "NC+NC"

## Understanding the Data

The data was downloaded when you setup your local environment and is now available in the `data/` directory.

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

In [7]:
def load_data(
    data_directory_path: str = "data",
):
    # If backed='r', load AnnData in backed mode instead of fully loading it into memory
    adata_train = scanpy.read_h5ad(os.path.join(data_directory_path, "obesity_challenge_2.h5ad"), backed="r")

    return adata_train

In [8]:
adata_train = load_data()

### Understanding `adata_train`

- The dataset is provided in [`AnnData` format](https://anndata.readthedocs.io/en/stable/) (.h5ad).

- Normalized gene expression values are stored in `adata.X` after per-cell total count normalization followed by $\log_2(1 + x)$ transformation (standard single-cell RNA-seq normalization; see [lecture 2 of the crash course](https://docs.crunchdao.com/competitions/competitions/broad-obesity/crash-course#lecture-2)).

- Raw gene expression counts prior to normalization are stored in `adata.layers['counts']` for reproducibility and alternative preprocessing.

- The perturbation target gene information is provided in `adata.obs['gene']`, with values corresponding to either `"NC"` for control cells or to the target gene name if the cell is perturbed. Control cells receive a perturbation that has no effect on the cell’s RNA-Seq profile.

- Cell state/program enrichment information is provided in `.obs`, with columns `pre_adipo`, `adipo`, `lipo`, and `other` indicating whether each cell was enriched for pre-adipocyte, adipocyte, or lipogenic programs. Other was defined as cells that were not enriched for either pre-adipocyte or adipocyte programs. **Program enrichment assignments were based on expert-curated canonical signature genes, and the list of signature genes is provided in `signature_genes.csv`.**

- We provide the cell state proportion for each of the perturbations in a separate file `program_proportion.csv`.

- During preprocessing, standard single-cell quality control (QC) was applied to remove low-quality cells and cell doublets based on sequencing library complexity, gene detection rate, and mitochondrial gene content. The dataset was then restricted to cells with a single confident guide assignment to a perturbation, and guides represented by fewer than 10 cells were excluded. Genes detected in fewer than 10 cells were removed, and known signature genes from `signature_genes.csv` were subsequently re-introduced.

**The .obs columns are defined as:**

- **cell index column**: The original sample ID.

- **nCount_RNA**: The number of Unique Molecular Identifiers (UMIs) detected per cell.

- **nFeature_RNA**: Number of genes with at least one detected UMI in the cell.

- **nCount_guide**: The number of single guide RNA (sgRNA) UMIs detected per cell.

- **nFeature_guide**: The number of sgRNAs detected per cell.

- **percent.mt**: The fraction of UMIs per cell that map to mitochondrial transcripts.

- **SampleID**: The sample ID.

- **Day**: The day of sample collection.

- **num_features**: The number of guides per cell (for low multiplicity of infection (MOI) data, after quality control, only the cells with 1 guide are kept).

- **feature_call**: The guide assignment of each cell.

- **num_umis**: The number of guide UMIs per cell.

- **gene**: **The perturbation single/double/triple target gene** (or perturbation identity).

- Cell state/program enrichment information is provided, with columns **pre_adipo**, **adipo**, **lipo**, and **other** indicating whether each cell was enriched for pre-adipocyte, adipocyte, or lipogenic programs.


**The "Cell Identity":**

Pre-adipocyte: Early-stage cells that haven't differentiated yet.

Adipocyte: Mature fat cells (the standard white fat).

Lipogenic: Specialized fat-producing cells.

Other: Cells that followed a different developmental path.

To understand how these **cell identity program proportions** were computed, please refer to the [program_analysis section in the GitHub repository](https://github.com/julielaffy/obesity-broad-ml-competition-2025?tab=readme-ov-file).

In [9]:
# Meta data for the cells in adata_train
# cells with "NC" or "NC+NC" in column "gene" are control cells (no gene perturbation)
adata_train.obs

,nCount_RNA,nFeature_RNA,nCount_guide,nFeature_guide,percent.mt,SampleID,Day,num_features,feature_call,num_umis,gene,adipo,pre_adipo,other,lipo
cell,,,,,,,,,,,,,,,
P10_AAACCCTGTCCCGAAC-1,8941,2854,15,3,3.646125,TF15_MOI2_10,Day14,1,NC-1,12,NC,0,1,0,0
P10_AAACGCCTCAGTACGC-1,13904,4118,17,4,6.918872,TF15_MOI2_10,Day14,2,NR3C1-3|STAT5B-6,6|9,NR3C1+STAT5B,1,0,0,1
P10_AAACGCCTCCCAGATA-1,13378,3723,12,2,3.692630,TF15_MOI2_10,Day14,2,NR3C1-1|NR3C1-9,8|4,NR3C1+NR3C1,0,0,1,0
P10_AAACGGACATCCGTAC-1,14924,3398,22,2,5.997052,TF15_MOI2_10,Day14,1,NC-5,21,NC,1,0,0,1
P10_AAACTCACAAGCTCGA-1,9669,3028,45,4,4.095563,TF15_MOI2_10,Day14,2,TCF7L2-8|ZBED3-7,3|40,TCF7L2+ZBED3,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P9_TGTGCGCGTAGTAACG-1,21186,4597,25,2,2.770698,TF15_MOI2_9,Day14,2,CEBPD-8|SF3B1-2,11|14,CEBPD+SF3B1,1,0,0,0
P9_TGTGGTCAGAAGCGTG-1,16835,4384,34,4,2.857143,TF15_MOI2_9,Day14,2,CEBPD-10|PPARG-10,26|6,CEBPD+PPARG,1,0,0,0
P9_TGTGGTCAGGTACGGA-1,18513,4160,28,2,2.933074,TF15_MOI2_9,Day14,2,FOXO1-6|TCF7L2-8,20|8,FOXO1+TCF7L2,0,0,1,0


Perturbation identity:

In the train set, there are single, double and triple gene perturbations.

In [10]:
adata_train.obs["gene"].unique()

['NC', 'NR3C1+STAT5B', 'NR3C1+NR3C1', 'TCF7L2+ZBED3', 'CEBPB+KLF15+POLR2D', ..., 'KIF11+KLF15+NR3C1', 'CEBPA+CEBPB+STAT5B', 'CEBPB+KIF11+KLF15', 'NR3C1+SF3B1+ZBED3', 'KIF11+NC+TCF7L2']
Length: 237
Categories (237, object): ['CEBPA+CEBPA', 'CEBPA+CEBPA+CEBPB', 'CEBPA+CEBPA+NR3C1',
                           'CEBPA+CEBPA+POLR2D', ..., 'TCF7L2+TCF7L2', 'TCF7L2+ZBED3', 'ZBED3',
                           'ZBED3+ZBED3']

## Subset of X_train

Normalized gene expression values are stored in `adata.X` after per-cell total count normalization followed by $\log_2(1 + x)$ transformation (standard single-cell RNA-seq normalization).
Use `adata.layers['counts']` instead of `adata.X` if you want to use unnormalized expression.

In [11]:
number_cells = 100
print(f"Subset of X_train: {number_cells} cells")

pd.DataFrame(
    adata_train.X[:number_cells, :].toarray(),
    index=adata_train.obs.index[:number_cells],
    columns=adata_train.var.index,
)

Subset of X_train: 100 cells


gene,INKA2,ETS2,AL512328.1,SMYD2,AP005900.1,EFR3B,CACNG1,LINC01449,BAZ1B,AC008687.3,...,ABO,ERVK-28,PSMD3,UBE2V1,VWDE,AC098649.1,SPAG7,ARMC12,OR51A2,RRP8
cell,,,,,,,,,,,,,,,,,,,,,
P10_AAACCCTGTCCCGAAC-1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,4.546516,0.0,...,0.0,0.0,0.000000,5.110751,0.0,0.0,0.000000,0.0,0.0,0.000000
P10_AAACGCCTCAGTACGC-1,0.0,0.0,0.0,3.034247,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,3.034247,3.034247,0.0,0.0,3.034247,0.0,0.0,0.000000
P10_AAACGCCTCCCAGATA-1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,3.995477,4.949527,0.0,0.0,0.000000,0.0,0.0,0.000000
P10_AAACGGACATCCGTAC-1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,4.797141,0.0,0.0,2.944974,0.0,0.0,2.944974
P10_AAACTCACAAGCTCGA-1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,5.001216,0.0,0.0,0.000000,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P10_AATGATGAGTGAAGCG-1,0.0,0.0,0.0,3.575963,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000
P10_AATGCCTGTTCCGCTT-1,0.0,0.0,0.0,2.716188,0.0,0.0,0.0,0.0,2.716188,0.0,...,0.0,0.0,0.000000,5.105512,0.0,0.0,0.000000,0.0,0.0,3.602015
P10_AATGGCAGTTGACCGC-1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,2.788502,4.932892,0.0,0.0,0.000000,0.0,0.0,0.000000


In [12]:
# Convert the `.X` matrix of the AnnData object to a dense NumPy array
def to_array(
    adata: anndata.AnnData,
) -> np.typing.NDArray[np.float64]:
    if isinstance(adata.X, np.ndarray):
        return adata.X

    return adata.X.toarray()

In [13]:
# If you want to convert to a Numpy array
# X_train_values = to_array(adata_train)
# X_train_values.shape

print(f"Train set: {adata_train.X.shape[0]} cells and {adata_train.X.shape[1]} gene columns")

adata_train.X

Train set: 23431 cells and 36601 gene columns


CSRDataset: backend hdf5, shape (23431, 36601), data_dtype float64

## Prepare Local Train/Test Datasets

To speed up experimentation, the **small** setting (`--size small`) uses a reduced dataset (~2 GB) and a predefined split:

- **Local train:** `obesity_challenge_1.h5ad` (23431 cells, 36601 gene columns and 235 unique perturbed genes) <br />
  Contains single-cell expression profiles for a subset of gene perturbations used for training.

- **Local test:** `obesity_challenge_2_local_gtruth.h5ad` <br />
  A held-out subset of perturbations used locally to evaluate model performance.

If you prefer, you can create your **own train/test split** or run **cross-validation** by switching to the full dataset:

- **Full dataset mode:** `--size default` (90815 cells, 36601 gene columns and 235 unique perturbed genes) <br />
  Downloads the entire available dataset (~8 GB), allowing you to design your own evaluation strategy.

In [14]:
def extract_unique_genes(gene_list, separator="+"):
    """
    Extract unique gene names from a list of strings.

    Parameters
    ----------
    gene_list : list of str
        List like ["GENE1+GENE2", "GENE3", ...]
    separator : str
        Separator between genes (default="+")

    Returns
    -------
    list
        List of unique gene names
    """
    return set({
        gene.strip()
        for item in gene_list
        for gene in item.split(separator)
        if gene.strip()
    })


def prepare_local_split_adata(
    adata,
    data_dir="data",
    single_control_label="NC",
    double_control_label="NC+NC",
    filter_genes=True,
    filter_cells=True,
    number_cells_per_gene=100,
    test_split_mode="fixed",
    fixed_test_genes=None,
    test_ratio=0.2,
    genes_to_predict_file="predict_genes_2.txt",
    perturbations_file="predict_perturbations_2.txt",
    random_seed=42,
):
    """
    Prepares local training and test AnnData objects.

    Parameters
    ----------
    adata : AnnData
        Input AnnData object.
    data_dir : str
        Directory to read/write data files.
    single_control_label : str
        Label for single control cells.
    double_control_label : str
        Label for double control cells.
    number_cells_per_gene : int
        Maximum number of cells to keep per gene.
    test_split_mode : str
        "fixed" or "ratio" for test/train split.
    fixed_test_genes : list
        List of test genes if test_split_mode="fixed".
    test_ratio : float
        Ratio of genes to keep as test if test_split_mode="ratio".
    genes_to_predict_file : str
        Filename for genes (columns) to predict (one gene per line).
    perturbations_file : str
        Filename for predicted perturbations (one gene per line).
    random_seed : int
        Seed for reproducible random splits.

    Returns
    -------
    local_adata_train : AnnData
        Filtered training AnnData.
    test_adata_eval : AnnData
        Filtered test AnnData with additional info in `uns`.
    test_genes: list
        List of test genes
    """

    if filter_genes:
        # Load predicted perturbations and genes to predict
        predict_perturbations = set(pd.read_csv(os.path.join(data_dir, perturbations_file), header=None)[0])
        observed_genes = extract_unique_genes(set(adata.obs["gene"])) - {single_control_label} - {double_control_label}
        print(len(observed_genes))
        print(observed_genes)

        # Keep genes in any of observed or predicted perturbations
        genes_to_keep = observed_genes | predict_perturbations
        genes_to_keep = list(genes_to_keep & set(adata.var.index))
    else:
        genes_to_keep = adata.var.index

    print("Number of genes kept:", len(genes_to_keep))

    if filter_cells:
        # Separate control and perturbed cells
        control_mask = adata.obs["gene"].isin([single_control_label, double_control_label])
        perturbed_mask = ~control_mask

        # Compute number of control cells to keep based on original ratio
        n_control_total = control_mask.sum()
        n_perturbed_total = perturbed_mask.sum()
        control_fraction = n_control_total / (n_control_total + n_perturbed_total)

        # Subsample perturbed cells: up to number_cells_per_gene per gene
        perturbed_cells_to_keep = adata.obs[perturbed_mask].groupby("gene", observed=True).head(number_cells_per_gene).index
        n_perturbed_keep = len(perturbed_cells_to_keep)
        n_control_keep = int(control_fraction * (n_perturbed_keep / (1 - control_fraction)))

        # Randomly sample control cells proportionally
        control_cells_to_keep = adata.obs[control_mask].sample(n=min(n_control_keep, n_control_total), random_state=42).index

        # Combine
        cells_to_keep = perturbed_cells_to_keep.union(control_cells_to_keep)
        print(f"Number of perturbed cells kept: {len(perturbed_cells_to_keep)}")
        print(f"Number of control cells kept: {len(control_cells_to_keep)}")
    else:
        cells_to_keep = adata.obs.index

    print("Total cells kept:", len(cells_to_keep))

    adata_filtered = adata[cells_to_keep, genes_to_keep].to_memory().copy()

    # Create train/test gene split
    genes = adata_filtered.obs["gene"].unique().tolist()
    if test_split_mode == "fixed":
        if fixed_test_genes is None:
            fixed_test_genes = []
        test_genes = fixed_test_genes
        train_genes = [g for g in genes if g not in test_genes]
    elif test_split_mode == "ratio":
        np.random.seed(random_seed)
        genes.remove(single_control_label)
        genes.remove(double_control_label)
        genes = np.random.permutation(genes).tolist()
        n_test = int(len(genes) * test_ratio)
        test_genes = genes[:n_test]
        train_genes = genes[n_test:] + [single_control_label, double_control_label]
    else:
        raise ValueError(f"Unknown test_split_mode: {test_split_mode}")

    print("Num train genes:", len(train_genes))
    print("Num test genes:", len(test_genes))

    # Filter cells by gene split
    local_adata_train = adata_filtered[adata_filtered.obs["gene"].isin(train_genes)].copy()
    local_adata_test = adata_filtered[adata_filtered.obs["gene"].isin(test_genes)].copy()

    # Convert X to dense if needed
    train_X = to_array(local_adata_train)
    test_X = to_array(local_adata_test)

    # Getting the control centroid in the train set
    single_control_mask = local_adata_train.obs["gene"] == single_control_label
    assert single_control_mask.sum() != 0, "No control cell"
    single_control_centroid = train_X[single_control_mask].mean(axis=0)

    double_control_mask = local_adata_train.obs["gene"] == double_control_label
    assert double_control_mask.sum() != 0, "No control cell"
    double_control_centroid = train_X[double_control_mask].mean(axis=0)

    # Getting the single perturbation mask
    single_perturb_mask = []
    double_perturb_mask = []
    for guide_gene in local_adata_train.obs["gene"]:
        # Getting the split names
        guide_elements = guide_gene.split("+")
        if len(guide_elements) == 1 and guide_gene != "NC":
            single_perturb_mask.append(True)
            double_perturb_mask.append(False)
        elif len(guide_elements) == 2 and guide_gene != "NC+NC":
            double_perturb_mask.append(True)
            single_perturb_mask.append(False)
        else:
            single_perturb_mask.append(False)
            double_perturb_mask.append(False)
    # Converting mask to series
    single_perturb_mask = pd.Series(single_perturb_mask, index=local_adata_train.obs["gene"]).values
    double_perturb_mask = pd.Series(double_perturb_mask, index=local_adata_train.obs["gene"]).values

    # Getting the perturbed centroid
    single_perturbed_centroid = train_X[single_perturb_mask].mean(axis=0)
    double_perturbed_centroid = train_X[double_perturb_mask].mean(axis=0)

    # Prepare test AnnData with additional info in `uns`
    test_adata_eval = anndata.AnnData(
        X=test_X,
        obs=local_adata_test.obs.copy(),
        uns=dict(
            single_control_centroid_train=single_control_centroid,
            double_control_centroid_train=double_control_centroid,
            single_perturbed_centroid_train=single_perturbed_centroid,
            double_perturbed_centroid_train=double_perturbed_centroid,
        )
    )

    print("local_adata_train:\n", local_adata_train)
    print("test_adata_eval:\n", test_adata_eval)

    # Save results
    local_adata_train.write(os.path.join(data_dir, "local_train.h5ad"))
    test_adata_eval.write(os.path.join(data_dir, "local_test_eval.h5ad"))
    with open(os.path.join(data_dir, "local_test_perturbated_genes.txt"), "w") as f:
        f.write("\n".join(test_genes))

    return local_adata_train, test_adata_eval, test_genes

You could use this to create your own split / cross-validation.

For this quickstarter, we don't, we will use the provided split:
- local train: `obesity_challenge_2.h5ad`
- and local test: `obesity_challenge_2_local_gtruth.h5ad`

In [15]:
# local_adata_train, test_adata_eval, test_genes = prepare_local_split_adata(
#     adata_train,
#     filter_genes=True,
#     filter_cells=True,
#     number_cells_per_gene=50,
#     test_split_mode="ratio",
#     test_ratio=0.2,
# )

## Strategy Implementation

In this section, we build a **simple baseline predictor** based on cell-type centroids computed from the training data.

The idea is straightforward:
- Compute the **control centroid** → average expression profile of all control cells.

- For each target perturbation in the evaluation set, generate synthetic "predicted cells" by **repeating the control centroid** a fixed number of times. **This means that perturbing this gene does not induce any change.**

- Package these predictions into an `AnnData` object with the correct structure.

In [16]:
def get_centroids(
    adata_train: anndata.AnnData,
    n_cells_per_perturbation: int | None = 50
):
    """
    Compute control and perturbed centroids
    """

    # Getting the control centroid in the train set
    print("Compute single_control_centroid")
    single_control_mask = adata_train.obs["gene"] == single_control_label
    assert single_control_mask.sum() != 0, "No single control cell"
    single_control_centroid = np.asarray(
        adata_train[single_control_mask].X.mean(axis=0)
    ).ravel()

    print("Compute double_control_centroid")
    double_control_mask = adata_train.obs["gene"] == double_control_label
    assert double_control_mask.sum() != 0, "No double control cell"
    double_control_centroid = np.asarray(
        adata_train[double_control_mask].X.mean(axis=0)
    ).ravel()

    # Getting the overall control centroid
    print("Compute overall_control_centroid")
    overall_control_mask = single_control_mask | double_control_mask
    overall_control_centroid = np.asarray(
        adata_train[overall_control_mask].X.mean(axis=0)
    ).ravel()

    # Getting the single and double perturbation mask
    single_perturb_mask = []
    double_perturb_mask = []
    for guide_gene in adata_train.obs["gene"]:
        # Getting the split names
        guide_elements = guide_gene.split("+")
        if len(guide_elements) == 1 and guide_gene != "NC":
            single_perturb_mask.append(True)
            double_perturb_mask.append(False)
        elif len(guide_elements) == 2 and guide_gene != "NC+NC":
            double_perturb_mask.append(True)
            single_perturb_mask.append(False)
        else:
            single_perturb_mask.append(False)
            double_perturb_mask.append(False)
    # Converting mask to series
    single_perturb_mask = pd.Series(single_perturb_mask, index=adata_train.obs_names)
    assert single_perturb_mask.sum() != 0, "No single perturbed cell"
    double_perturb_mask = pd.Series(double_perturb_mask, index=adata_train.obs_names)
    assert double_perturb_mask.sum() != 0, "No double perturbed cell"

    if n_cells_per_perturbation is not None:
        # Create a reduced index by sampling N cells per perturbation
        single_perturb_mask = single_perturb_mask.index.isin(
            adata_train.obs.loc[single_perturb_mask]
            .groupby("gene", observed=True)
            .head(n_cells_per_perturbation)
            .index
        )
        double_perturb_mask = double_perturb_mask.index.isin(
            adata_train.obs.loc[double_perturb_mask]
            .groupby("gene", observed=True)
            .head(n_cells_per_perturbation)
            .index
        )
    else:
        single_perturb_mask = single_perturb_mask.values
        double_perturb_mask = double_perturb_mask.values

    # Getting the control centroid in the train set
    print("Compute single_perturbed_centroid")
    single_perturbed_centroid = np.asarray(
        adata_train[single_perturb_mask].X.mean(axis=0)
    ).ravel()

    print("Compute double_perturbed_centroid")
    double_perturbed_centroid = np.asarray(
        adata_train[double_perturb_mask].X.mean(axis=0)
    ).ravel()

    # Getting the overall perturbed centroid
    print("Compute overall_perturbed_centroid")
    overall_perturb_mask = single_perturb_mask | double_perturb_mask
    overall_perturbed_centroid = np.asarray(
        adata_train[overall_perturb_mask].X.mean(axis=0)
    ).ravel()

    gc.collect()

    return single_control_mask, double_control_mask, overall_control_mask, \
        single_perturb_mask, double_perturb_mask, overall_perturb_mask, \
        single_control_centroid, double_control_centroid, overall_control_centroid, \
        single_perturbed_centroid, double_perturbed_centroid, overall_perturbed_centroid

We compute reference profiles from the training set:

- **Single Control centroid**: mean expression across all single control cells
- **Single Perturbed centroid**: mean expression across all single non-control (perturbed) cells
- **Double Control centroid**: mean expression across all double control cells
- **Double Perturbed centroid**: mean expression across all double non-control (perturbed) cells
- **Overall Control centroid**: mean expression across all control cells
- **Overall Perturbed centroid**: mean expression across all non-control (perturbed) cells

In [17]:
(
    single_control_mask,
    double_control_mask,
    overall_control_mask,
    single_perturb_mask,
    double_perturb_mask,
    overall_perturb_mask,

    single_control_centroid,
    double_control_centroid,
    overall_control_centroid,
    single_perturbed_centroid,
    double_perturbed_centroid,
    overall_perturbed_centroid,
) = get_centroids(
    adata_train,
    n_cells_per_perturbation=50
)

print("double_control_centroid shape:", double_control_centroid.shape)
print("double_perturbed_centroid shape:", double_perturbed_centroid.shape)

Compute single_control_centroid
Compute double_control_centroid
Compute overall_control_centroid
Compute single_perturbed_centroid
Compute double_perturbed_centroid
Compute overall_perturbed_centroid
double_control_centroid shape: (36601,)
double_perturbed_centroid shape: (36601,)


Load the list of gene perturbations to predict.

For the local version, we evaluate on a separate test file (`"obesity_challenge_2_local_gtruth.h5ad"`):

In [18]:
# For your local evaluation, we use a separate test file with ground truth data.
# The file contains a subset of genes for which we want to predict the perturbation effects.
gtruth = scanpy.read_h5ad(os.path.join("data", "obesity_challenge_2_local_gtruth.h5ad"), backed="r")
predict_perturbations = gtruth.obs["gene"].cat.categories.tolist()

print("Local test gene perturbations:", predict_perturbations)

Local test gene perturbations: ['CEBPA', 'CEBPB+KLF15', 'CEBPB+MLXIPL', 'FOXO1+SF3B1', 'KLF15', 'KLF15+MLXIPL']


In [19]:
gtruth

AnnData object with n_obs × n_vars = 1800 × 36601 backed at 'data/obesity_challenge_2_local_gtruth.h5ad'
    obs: 'gene'
    uns: 'double_control_centroid_train', 'double_perturbed_centroid_train', 'single_control_centroid_train', 'single_perturbed_centroid_train'

In the full challenge, you would load the 62 unseen perturbations via `predict_perturbations_2.txt`:

In [20]:
# predict_perturbations = (
#     pd.read_csv(os.path.join("data", "predict_perturbations_2.txt"), header=None)[0]
#     .values
# )

# print("All gene perturbations:", predict_perturbations)

Generate predictions using the control centroid baseline.

We create a prediction matrix where each gene perturbation is represented by `cells_per_perturbation` (100) identical rows equal to the single or double control centroid.

In [22]:
def fill_X(
    X, predict_perturbations: List[str],
    genes_to_predict: List[str],
    methods: Dict[str, str],
    single_control_centroid,
    double_control_centroid,
    overall_control_centroid,
    single_perturbed_centroid,
    double_perturbed_centroid,
    overall_perturbed_centroid,
    cells_per_perturbation,
    train_std,
    add_noise,
):
    """
    Fill the prediction matrix with synthetic single-cell gene expression profiles.
    For each perturbation, the function repeats the perturbation-level predicted
    gene expression and optionally adds gene-wise Gaussian noise to simulate
    cell-to-cell variability.
    """

    np.random.seed(42)
    for i, pert in tqdm(enumerate(predict_perturbations), total=len(predict_perturbations), desc="Generating synthetic cells"):
        start = i * cells_per_perturbation
        end = (i + 1) * cells_per_perturbation

        # Getting the split names
        guide_elements = pert.split("+")

        if len(guide_elements) == 1:
            if methods["single"] == "control":
                pred_profile = single_control_centroid
            elif methods["single"] == "perturbed":
                pred_profile = single_perturbed_centroid
            else:
                raise NotImplementedError("Method not implemented")
        elif len(guide_elements) == 2:
            if methods["double"] == "double_control":
                pred_profile = double_control_centroid
            elif methods["double"] == "overall_control":
                pred_profile = overall_control_centroid
            elif methods["double"] == "double_perturbed":
                pred_profile = double_perturbed_centroid
            elif methods["double"] == "overall_perturbed":
                pred_profile = overall_perturbed_centroid
            else:
                raise NotImplementedError("Method not implemented")
        else:
            raise ValueError("Validation set contains perturbation which is not allowed")

        # Optionally add gene-wise Gaussian noise
        if add_noise:
            noise = np.random.normal(
                loc=0.0,
                scale=train_std,
                size=(cells_per_perturbation, len(genes_to_predict))
            )
            pred_profile = pred_profile + noise

        X[start:end] = pred_profile.astype(np.float32)

In [23]:
# GENERATE PREDICTIONS
genes_to_predict = list(adata_train.var.index)
n_genes = len(genes_to_predict)
n_perturbations = len(predict_perturbations)
cells_per_perturbation = 100

methods = {
    "single": "control",  # "control" or "perturbed"
    "double": "double_control"  # "double_control" or "double_perturbed" or "overall_control" or "overall_perturbed"
}

# Getting the std-dev for each gene in the train set
train_std = None  # to_array(adata_train).std(axis=0) # remove backed="r" mode to compute std
# Whether to add Gaussian noise based on training gene-wise std:
add_train_std = False

# Initialize prediction matrix
# (n_perturbations * cells_per_pert, n_genes) prediction matrix
prediction_matrix = np.zeros((n_perturbations * cells_per_perturbation, n_genes))

# Fill prediction_matrix with a repeated centroid depending on chosen methods
fill_X(prediction_matrix, predict_perturbations, genes_to_predict, methods,
       single_control_centroid, double_control_centroid, overall_control_centroid,
       single_perturbed_centroid, double_perturbed_centroid, overall_perturbed_centroid,
       cells_per_perturbation, train_std, add_train_std)

# ------------------------------------------------------
# Construct obs["gene"] for the AnnData object
# Each perturbation label is repeated 'cells_per_perturbation' times
# Total rows = n_perturbations * cells_per_perturbation
# Example: 5 perturbations × 100 cells -> 500 rows
# ------------------------------------------------------
obs_gene = np.repeat(predict_perturbations, cells_per_perturbation)

# Build the AnnData output object
prediction = anndata.AnnData(
    X=prediction_matrix,
    obs={"gene": obs_gene},
    var=adata_train.var.copy(),   # To preserve gene names & order
)

# The resulting object contains synthetic single-cell predictions
# with one row per cell and one column per gene
prediction

Generating synthetic cells:   0%|          | 0/6 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 600 × 36601
    obs: 'gene'

### Why do we generate 100 predicted cells per perturbation?

Single-cell RNA-seq produces **many individual cells per perturbation**, not a single expression vector. Even when the same gene is perturbed, cells show diverse responses:
- different effect magnitudes,
- alternative differentiation paths,
- alternative metabolic states,
- natural transcriptional variability.

This heterogeneity is real **biological signal**, not noise, and the challenge evaluates how well a model reproduces the distribution of cell states.

The two evaluation metrics reflect this: **MMD** compares full distributions using pairwise similarities between cells and **Pearson Delta** evaluates perturbation effects relative to perturbed means. Predicting only one cell would collapse the distribution, give unstable or biased estimates, and distort correlations.

By generating **100 (could be more) predicted cells per perturbation**, it create enough samples to capture heterogeneity, support distribution-based metrics, stabilize mean/variance estimates and realistically approximate the “cloud” of single-cell responses observed in real data.

### Predict Program Proportions

For this baseline, we simply copy the mean program proportions of control cells from the training set.

In [28]:
def estimate_cell_type_proportions(
    adata_train,
    predict_perturbations: List[str],
    methods: Dict[str, str],
    single_control_mask: pd.Series,
    single_perturb_mask: pd.Series,
    double_control_mask: pd.Series,
    overall_control_mask: pd.Series,
    double_perturb_mask: pd.Series,
    overall_perturb_mask: pd.Series,
) -> pd.DataFrame:
    """
    Estimate cell-type proportions for each perturbation using the specified masks.

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by perturbation name with columns:
        ['gene', 'adipo', 'pre_adipo', 'other', 'lipo']
    """
    pred_proportion_dict = {}

    for pert in tqdm(predict_perturbations):
        guide_elements = pert.split("+")
        proportion_mask = None

        # Single perturbation
        if len(guide_elements) == 1:
            if methods["single"] == "control":
                proportion_mask = single_control_mask
            elif methods["single"] == "perturbed":
                proportion_mask = single_perturb_mask
            else:
                raise NotImplementedError("Single method not implemented")

        # Double perturbation
        elif len(guide_elements) == 2:
            if methods["double"] == "double_control":
                proportion_mask = double_control_mask
            elif methods["double"] == "overall_control":
                proportion_mask = overall_control_mask
            elif methods["double"] == "double_perturbed":
                proportion_mask = double_perturb_mask
            elif methods["double"] == "overall_perturbed":
                proportion_mask = overall_perturb_mask
            else:
                raise NotImplementedError("Double method not implemented")

        else:
            raise ValueError(
                "Validation set contains perturbation which is not allowed"
            )

        adata_train_perturb = adata_train[proportion_mask]

        pred_proportion_dict[pert] = dict(
            adipo=float(adata_train_perturb.obs["adipo"].mean()),
            pre_adipo=float(adata_train_perturb.obs["pre_adipo"].mean()),
            other=float(adata_train_perturb.obs["other"].mean()),
            lipo=float(adata_train_perturb.obs["lipo"].mean()),
        )

    pred_proportion_df = pd.DataFrame.from_dict(
        pred_proportion_dict, orient="index"
    )
    pred_proportion_df["gene"] = pred_proportion_df.index

    return pred_proportion_df


def postprocess_proportions(pred_proportion_df: pd.DataFrame) -> pd.DataFrame:
    """
    Post-process predicted proportions:
    - Normalize (pre_adipo + adipo + other) to sum to 1
    - Ensure lipo <= adipo
    """

    pred_proportion_df = pred_proportion_df.copy()

    # Normalize selected columns to sum to 1
    cols = ["pre_adipo", "adipo", "other"]
    pred_proportion_df[cols] = (
        pred_proportion_df[cols]
        .div(pred_proportion_df[cols].sum(axis=1), axis=0)
    )

    # Enforce lipo <= adipo
    pred_proportion_df["lipo"] = pred_proportion_df[
        ["lipo", "adipo"]
    ].min(axis=1)

    return pred_proportion_df

In [29]:
pred_proportion_df = estimate_cell_type_proportions(
    adata_train,
    predict_perturbations,
    methods,
    single_control_mask, single_perturb_mask, double_control_mask,
    overall_control_mask, double_perturb_mask, overall_perturb_mask,
)

pred_proportion_df = postprocess_proportions(pred_proportion_df)
pred_proportion_df

  0%|          | 0/6 [00:00<?, ?it/s]

,adipo,pre_adipo,other,lipo,gene
CEBPA,0.260046,0.272171,0.467783,0.066744,CEBPA
CEBPB+KLF15,0.357143,0.285714,0.357143,0.107143,CEBPB+KLF15
CEBPB+MLXIPL,0.357143,0.285714,0.357143,0.107143,CEBPB+MLXIPL
FOXO1+SF3B1,0.357143,0.285714,0.357143,0.107143,FOXO1+SF3B1
KLF15,0.260046,0.272171,0.467783,0.066744,KLF15
KLF15+MLXIPL,0.357143,0.285714,0.357143,0.107143,KLF15+MLXIPL


### The `train()` Function

The `train()` function is intended to encapsulate the full training workflow used to fit a predictive model on the provided perturbation dataset.
In a complete implementation, this function would:

- Load the training dataset from `data_directory_path`
- Preprocess the gene expression matrix
- Construct and train a model
- Evaluate the model on a validation split
- Save the trained model and any required metadata to `model_directory_path`

**On the platform, `train()` will be run on the full large dataset `obesity_challenge_2.h5ad`.**

**Note**:
- ❗ We recommend training locally and submitting weights/models because the dataset is large and cloud resources are limited.
- Make sure that the `Method description.md` file properly documents your model, so that the Broad Institute team can reference your work in their publications.

In [30]:
def train(
    data_directory_path: str,
    model_directory_path: str,
):
    """
    Train a perturbation prediction model.

    This function is designed to:
      - Load training data from `data_directory_path`
      - Preprocess gene expression matrices
      - Train your model on the available perturbations
      - Save all required models into `model_directory_path`

    In this baseline notebook, the function is left empty.
    You can fill in the model pipeline you want to use.
    """

    pass

### The `infer()` Function

The `infer()` function performs model inference on a set of unseen perturbations.
In the context of this baseline solution, there is no trained model.
Instead, predictions are generated by reusing the control centroid, making this a simple but valid baseline for evaluating the metric computation pipeline.

During inference, the function:

1. Loads the training dataset.
2. Baseline strategy: Computes centroids for control and perturbed cells in the training data. Builds a prediction matrix by repeating the control centroid for every requested perturbation.
3. Constructs an `AnnData` object containing the predicted gene-expression profiles.
4. Saves the predictions in the correct structure expected by the specification.
5. Computes the predicted cell-type proportions and saves them as a CSV file.

[Expected Output](https://docs.crunchdao.com/competitions/competitions/broad-obesity/crunch-1#expected-output):

- `Method description.md`: At the __end of the notebook__ 👇👇👇, write a small text outlining the approaches used to generate both the predictions and the estimated proportions of cells enriched for each program.

- `prediction.h5ad`: An `AnnData` file containing predicted gene expression profiles normalized and log-transformed post-perturbation for 62 gene perturbations indicated in `predict_perturbations_2.txt`. Predictions should be stored in `adata.X` matrix with the corresponding perturbation identity recorded in `adata.obs['gene']`. For each gene perturbation, we ask you to predict the gene expression profiles for 100 cells to quantify the distribution of each perturbation prediction. The file with predictions should have dimensions [6,200 × len(genes_to_predict)] (cells × genes).

- `predict_program_proportion.csv`: A CSV file reporting the predicted proportion of cells with enriched programs for each gene perturbation listed in `predict_perturbations.txt`.

**Inference input:**
- `predict_perturbations`: A list of **single-gene perturbations** for which you must predict the transcriptomic effect.
- `genes_to_predict`: A list of **gene names (columns)** to include in your predictions. Your model must generate expression values for this specific set of genes and the order of columns in the final prediction file must follow this list.

**Provide an inference function that relies on these two inputs.**

In [31]:
def infer(
    data_directory_path: str,
    prediction_directory_path: str,
    prediction_h5ad_file_path: str,
    program_proportion_csv_file_path: str,
    model_directory_path: str,
    predict_perturbations: typing.List[str],
    genes_to_predict: typing.List[str],
    cells_per_perturbation: int = 100,
):
    """
    Run inference for a set of perturbations.

    Parameters:
        data_directory_path: str
            Path to the training AnnData file.
        prediction_directory_path: str
            Directory where prediction files can be written.
        prediction_h5ad_file_path: str
            Direct path where to write the `prediction.h5ad` file.
        program_proportion_csv_file_path: str
            Direct path where to write the `predict_program_proportion.csv` file.
        model_directory_path: str
            Directory containing your persisted model files.
        predict_perturbations: List[str]
            The perturbations for which to generate predictions.
        genes_to_predict: List[str]
            List of gene names (columns) to include in the prediction.h5ad AnnData object.
        cells_per_perturbation: int
            Number of synthetic cells to generate per perturbation.

    Return:
      None: Returned value is ignored.

    Expected files:
        prediction_directory_path / "prediction.h5ad": anndata.AnnData
            AnnData matrix of size (n_perturbations * cells_per_perturbation, n_genes_to_predict), containing the predicted gene expression values.
        prediction_directory_path / "predict_program_proportion.csv": pd.DataFrame
            DataFrame (index=False) containing estimated cell-type proportions for each perturbation.
    """

    print("Loading data...")
    global adata_train
    if "adata_train" not in globals():
        # Optimization for Google Colab, avoid loading the data twice
        print("load adata_train")
        adata_train = load_data(data_directory_path)

    methods = {
        "single": "control",  # "control" or "perturbed"
        "double": "double_control"  # "double_control" or "double_perturbed" or "overall_control" or "overall_perturbed"
    }

    print(
        "###############################################################\n"
        "# Gene profiles\n"
        "###############################################################"
    )
    print("Extracting centroids...")
    (
        single_control_mask,
        double_control_mask,
        overall_control_mask,
        single_perturb_mask,
        double_perturb_mask,
        overall_perturb_mask,

        single_control_centroid,
        double_control_centroid,
        overall_control_centroid,
        single_perturbed_centroid,
        double_perturbed_centroid,
        overall_perturbed_centroid,
    ) = get_centroids(
        adata_train,
        n_cells_per_perturbation=50
    )

    # Getting the std-dev for each gene in the train set
    train_std = None  # to_array(adata_train).std(axis=0) # remove backed="r" mode to compute std
    # Whether to add Gaussian noise based on training gene-wise std:
    add_train_std = False

    print("Filtering data to only genes to predict...")
    genes_to_predict_mask = adata_train.var.index.isin(genes_to_predict)
    single_control_centroid = single_control_centroid[genes_to_predict_mask]
    double_control_centroid = double_control_centroid[genes_to_predict_mask]
    overall_control_centroid = overall_control_centroid[genes_to_predict_mask]
    single_perturbed_centroid = single_perturbed_centroid[genes_to_predict_mask]
    double_perturbed_centroid = double_perturbed_centroid[genes_to_predict_mask]
    overall_perturbed_centroid = overall_perturbed_centroid[genes_to_predict_mask]
    if train_std is not None:
        train_std = train_std[genes_to_predict_mask]

    print("Infering the prediction...")
    n_genes = len(genes_to_predict)
    n_perturbations = len(predict_perturbations)
    n_cells = n_perturbations * cells_per_perturbation

    print(f"Predicting {n_cells} cells for {n_perturbations} perturbations.")
    print("Each row will contain", n_genes, "genes.")

    # Construct obs["gene"] (n_perturbations * cells_per_pert rows)
    obs = {"gene": np.repeat(predict_perturbations, cells_per_perturbation)}
    # Column variables
    var = adata_train[:, genes_to_predict].var.copy()

    # Clean any prior prediction file
    if os.path.exists(prediction_h5ad_file_path):
        os.remove(prediction_h5ad_file_path)

    # Temporary disk-mapped matrix file (will be removed after final .h5ad is written)
    temporary_prediction_path = os.path.join(prediction_directory_path, "prediction_matrix.h5")
    if os.path.exists(temporary_prediction_path):
        os.remove(temporary_prediction_path)

    # RAM estimation (informational)
    needed_ram = (n_cells * n_genes * 4) / (1024**3)  # GB
    available_ram = psutil.virtual_memory().available / (1024**3)
    print(f"Available RAM: {available_ram:.2f} GB")
    print(f"Needed RAM for full matrix: ~{needed_ram:.2f} GB")

    # Detecting whether you are running inside the crunch platform or not
    use_ram_intensive_mode = crunch.is_inside_runner
    # use_ram_intensive_mode = True  # Uncomment me to force it

    if use_ram_intensive_mode:
        print("-> Using full in-memory matrix (fastest).")

        # Full RAM X
        X = np.zeros((n_cells, n_genes), dtype=np.float32)
        print("Prediction matrix shape:", X.shape)

        fill_X(X, predict_perturbations, genes_to_predict, methods,
               single_control_centroid, double_control_centroid, overall_control_centroid,
               single_perturbed_centroid, double_perturbed_centroid, overall_perturbed_centroid,
               cells_per_perturbation, train_std, add_train_std)

        prediction = anndata.AnnData(X=X, obs=obs, var=var)
        del X

        # Save to .h5ad
        print("Saving the prediction...")
        prediction.write_h5ad(prediction_h5ad_file_path)
    else:
        print("-> Using HDF5 (low-memory mode).")
        # Adjust batch size
        batch_size = cells_per_perturbation  # 1024 if n_cells > 5000 else 50

        # Create an HDF5 dataset on disk and write predictions batch-by-batch.
        # This avoids storing a huge (e.g. 286,300 × len(genes_to_predict)) matrix in RAM.
        with h5py.File(temporary_prediction_path, "w") as f:
            print("Prediction matrix shape:", (n_cells, n_genes))
            dset = f.create_dataset(
                "X",
                shape=(n_cells, n_genes),
                dtype="float32",
                chunks=(batch_size, n_genes),
                # compression="gzip"
            )

            fill_X(dset, predict_perturbations, genes_to_predict, methods,
                   single_control_centroid, double_control_centroid, overall_control_centroid,
                   single_perturbed_centroid, double_perturbed_centroid, overall_perturbed_centroid,
                   cells_per_perturbation, train_std, add_train_std)

        print("Finished writing matrix.")

        # Use an empty in-memory array with the same shape, then replace X with the HDF5 dataset
        prediction = anndata.AnnData(X=np.zeros((n_cells, n_genes), dtype=np.float32), obs=obs, var=var)

        with h5py.File(temporary_prediction_path, "r") as f:
            # Point X to the HDF5 file
            prediction.X = f["X"]

            # Save to .h5ad
            print("Saving the prediction...")
            prediction.write_h5ad(prediction_h5ad_file_path)

        # Remove the temporary HDF5 file
        if os.path.exists(temporary_prediction_path):
            os.remove(temporary_prediction_path)

    print(
        "###############################################################\n"
        "# Program Proportions\n"
        "###############################################################"
    )

    print("Infering the proportion...")
    pred_proportion_df = estimate_cell_type_proportions(
        adata_train,
        predict_perturbations,
        methods,
        single_control_mask, single_perturb_mask, double_control_mask,
        overall_control_mask, double_perturb_mask, overall_perturb_mask,
    )

    pred_proportion_df = postprocess_proportions(pred_proportion_df)

    # Saving the prediction as csv
    print("Saving the proportion...")
    pred_proportion_df.to_csv(program_proportion_csv_file_path, index=False)

    del prediction
    gc.collect()

    print("Finished !")

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch_tools.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

**Note**: Locally, the infer function will be run on only 6 perturbated genes.

In [ ]:
crunch_tools.test()

In [33]:
genes_to_predict = pd.read_csv(os.path.join("data", "predict_genes_2.txt"), header=None)[0].values

# If instead you want to predict all genes (columns):
# genes_to_predict = adata_train.var.index.to_list()

print("About to predict", len(genes_to_predict), "genes")

About to predict 36601 genes


In [ ]:
# In the full challenge, you would load the 62 unseen perturbations via predict_perturbations.txt.
# predict_perturbations = (
#     pd.read_csv(os.path.join("data", "predict_perturbations_2.txt"), header=None)[0]
#     .values
# )

# Test infer function on local test set "obesity_challenge_2_local_gtruth.h5ad"
gtruth = scanpy.read_h5ad(os.path.join("data", "obesity_challenge_2_local_gtruth.h5ad"), backed="r")
predict_perturbations = gtruth.obs["gene"].cat.categories.tolist()
print("Local test gene perturbations:", predict_perturbations)

infer(
    data_directory_path="data/",
    prediction_directory_path="prediction/",
    model_directory_path="resources/",
    predict_perturbations=predict_perturbations,
    genes_to_predict=genes_to_predict,
    prediction_h5ad_file_path=os.path.join("prediction", "prediction.h5ad"),
    program_proportion_csv_file_path=os.path.join("prediction", "predict_program_proportion.csv")
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

In [35]:
prediction = scanpy.read_h5ad(os.path.join("prediction", "prediction.h5ad"))
prediction

AnnData object with n_obs × n_vars = 600 × 36601
    obs: 'gene'

In [36]:
predicted_proportion = pd.read_csv(os.path.join("prediction", "predict_program_proportion.csv"))
predicted_proportion

,adipo,pre_adipo,other,lipo,gene
0,0.260046,0.272171,0.467783,0.066744,CEBPA
1,0.357143,0.285714,0.357143,0.107143,CEBPB+KLF15
2,0.357143,0.285714,0.357143,0.107143,CEBPB+MLXIPL
3,0.357143,0.285714,0.357143,0.107143,FOXO1+SF3B1
4,0.260046,0.272171,0.467783,0.066744,KLF15
5,0.357143,0.285714,0.357143,0.107143,KLF15+MLXIPL


### Local scoring

You can call the function that the system uses to estimate your score locally.

In [37]:
gtruth = scanpy.read_h5ad(os.path.join("data", "obesity_challenge_2_local_gtruth.h5ad"))
gtruth

AnnData object with n_obs × n_vars = 1800 × 36601
    obs: 'gene'
    uns: 'double_control_centroid_train', 'double_perturbed_centroid_train', 'single_control_centroid_train', 'single_perturbed_centroid_train'

In [38]:
gtruth_proportion = pd.read_csv(os.path.join("data", "program_proportion_local_gtruth.csv"))
gtruth_proportion

,gene,pre_adipo,adipo,other,lipo,lipo_adipo
0,CEBPA,0.307291,0.168643,0.524067,0.031233,0.185205
1,KLF15,0.250248,0.295986,0.453766,0.087145,0.294422
2,FOXO1+SF3B1,0.136752,0.540598,0.322650,0.096154,0.177866
3,CEBPB+KLF15,0.363458,0.121807,0.514735,0.015717,0.129032
4,KLF15+MLXIPL,0.342342,0.150150,0.507508,0.024024,0.160000
5,CEBPB+MLXIPL,0.267409,0.169916,0.562674,0.025070,0.147541


### Transcriptome-wide metrics

These metrics are computed using **either a subset of genes or all genes** (i.e., columns of the predicted matrix) for each perturbation.

- For the **public leaderboard** (updated weekly), the evaluation uses **1,000 hidden genes**.
The **MMD metric** is computed with a fixed sigma value of **2326**, calibrated specifically for 1,000-dimensional gene vectors.
- For the **private leaderboard**, both the **number** and the **identity** of the scoring genes will remain unknown to participants.

In this quickstarter example, we follow the same spirit and randomly select 1,000 genes to compute transcriptome-wide evaluation metrics.

In [39]:
# Retrieve the ordered list of predicted gene names (columns)
genes_to_predict_prediction = prediction.var.index.tolist()

print("Number of genes (columns) in the prediction:", len(genes_to_predict_prediction))

# Number of genes to use for computing metrics
n_genes_for_metric = 1000

# Randomly sample genes for the metric computation
rng = np.random.default_rng(seed=42)  # You should try with different seed to make sure you are ready for the private leaderboard
genes_for_metric = rng.choice(genes_to_predict_prediction, size=n_genes_for_metric, replace=False).astype(str)

# Index for selecting the same genes from adata_train and local_gtruth
indexer_genes_gtruth = adata_train.var.index.get_indexer(genes_for_metric)

print("Number of genes (columns) used to compute metrics:", len(genes_for_metric))

Number of genes (columns) in the prediction: 36601
Number of genes (columns) used to compute metrics: 1000


#### Pearson Delta

The Pearson Delta ($\rho$) is the perturbation effects relative to perturbed mean ($\hat{X}_P$) between predicted ($\hat{X}$) and observed ($X$):

$
\displaystyle
\rho(X, \hat{X}) = \frac{1}{|P|} \sum_{p \in P} \text{cor}\left(\hat{X}_p - X_{PM}, X_p - X_{PM}\right)
$

where:
- $X_{PM}$ is the Perturbed Mean defined as the mean gene expression of all the perturbed cells (those receiving a single gene perturbation) in the training dataset,
- $P$ is the set of all perturbation targets,
- $|P|$ is the size of the set $P$,
- $\text{cor}(a, b)$ is the correlation between vectors $a$ and $b$.

---

*Higher is better!*

In [40]:
def compute_metric_pearson(
    gtruth_X: np.typing.NDArray[np.float64],
    pred_X: np.typing.NDArray[np.float64],
    perturbed_centroid: np.typing.NDArray[np.float64],
) -> float:
    gtruth_X_target = gtruth_X.mean(axis=0)
    pred_X_target = pred_X.mean(axis=0)

    return scipy.stats.pearsonr(
        gtruth_X_target - perturbed_centroid,
        pred_X_target - perturbed_centroid,
    ).statistic

perturbations = gtruth.obs["gene"].cat.categories.tolist()
scores_pearson = []

for p in tqdm(perturbations):
    # Select true and predicted cells for perturbation p
    gt_mask = gtruth.obs["gene"] == p
    pr_mask = prediction.obs["gene"] == p

    # Extract their filtered expression matrices
    gtruth_X = to_array(gtruth[gt_mask, genes_for_metric])
    pred_X   = to_array(prediction[pr_mask, genes_for_metric])

    # Skip perturbations without matching predicted cells
    if gtruth_X.shape[0] == 0 or pred_X.shape[0] == 0:
        print(f"skipping {p}: missing samples: will not be accepted on the platform!")
        scores_pearson.append(0)
        continue

    # perturbed_centroid is different depending on single/double perturbation
    guide_elements = p.split("+")
    if len(guide_elements)==1:
        perturbed_centroid = gtruth.uns["single_perturbed_centroid_train"][indexer_genes_gtruth]
    elif len(guide_elements)==2:
        perturbed_centroid = gtruth.uns["double_perturbed_centroid_train"][indexer_genes_gtruth]

    # Compute Pearson score for this perturbation
    score = compute_metric_pearson(
        gtruth_X=gtruth_X,
        pred_X=pred_X,
        perturbed_centroid=perturbed_centroid,
    )
    scores_pearson.append(score)

print(f"Pearson score: {np.mean(scores_pearson):.4f}")

  0%|          | 0/6 [00:00<?, ?it/s]

Pearson score: 0.0297


#### Maximum Mean Discrepancy (MMD)

The MMD is between the predicted and the observed distributions of single-cell profiles.

Let $X_p^i$ be the true expression profile and $\hat{X}_p^i$ be the predicted expression profile for cell $i$ with perturbation $p$.
  Then we calculate the MMD distance for a particular perturbation using the following formulae:

$
\displaystyle
MMD^2(X_p, \hat{X}_p) =
\frac{1}{N^2} \sum_{i,j} k(X_p^i, X_p^j) +
\frac{1}{N^2} \sum_{k,l} k(\hat{X}_p^k, \hat{X}_p^l) -
\frac{2}{N^2} \sum_{m,n} k(X_p^m, \hat{X}_p^n)
$

where:
- $k(a, b)$ is the Gaussian kernel with the bandwidth from the following list: $[581.5, 1163.0, 2326.0, 4652.0, 9304.0]$.

---

Then we average the MMD score across all the perturbations using the following formulae:

$
\displaystyle
MMD^2(X, \hat{X}) = \frac{1}{|P|} \sum_{p \in P} MMD^2(X_p, \hat{X}_p)
$

where:
- $|P|$ is the size of the set $P$.

---

*Lower is better!*

In [41]:
def _gaussian_kernel(source, target, kernel_mul, kernel_num, fix_sigma):
    # Getting the L2 distance
    n_samples = int(source.shape[0]) + int(target.shape[0])
    total = np.concatenate([source, target], axis=0)

    total0 = np.broadcast_to(np.expand_dims(total, axis=0), (int(total.shape[0]), int(total.shape[0]), int(total.shape[1])))
    total1 = np.broadcast_to(np.expand_dims(total, axis=1), (int(total.shape[0]), int(total.shape[0]), int(total.shape[1])))
    L2_distance = np.sum((total0 - total1)**2, axis=2)

    # Now we are ready to scale this using multiple bandwidth
    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = np.sum(L2_distance) / (n_samples**2 - n_samples)

    # Now we will create the multiple bandwidth list
    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul**i) for i in range(kernel_num)]
    kernel_val = [np.exp(-L2_distance / bandwidth_temp) for bandwidth_temp in bandwidth_list]

    return sum(kernel_val)


def _compute_mmd(X_batch, Y_batch, kernel_mul, kernel_num, fix_sigma, kernel_func):
    num_batch_element = X_batch.shape[0]

    kernels = kernel_func(X_batch, Y_batch, kernel_mul, kernel_num, fix_sigma)
    XX = kernels[:num_batch_element, :num_batch_element]
    YY = kernels[num_batch_element:, num_batch_element:]
    XY = kernels[:num_batch_element, num_batch_element:]
    YX = kernels[num_batch_element:, :num_batch_element]
    mmd_val = np.sum(XX + YY - XY - YX)

    return mmd_val / num_batch_element ** 2


def balance_source_target_sample_per_perturbation(gtruth_X_tgt,pred_X_tgt):
    num_gtruth = gtruth_X_tgt.shape[0]
    num_pred = pred_X_tgt.shape[0]
    min_sample = min(num_gtruth,num_pred)

    return gtruth_X_tgt[0:min_sample,:], pred_X_tgt[0:min_sample,:]


def compute_metric_mmd(
    gtruth_X: np.typing.NDArray[np.float64],
    pred_X: np.typing.NDArray[np.float64],
) -> float:
    kernel_mul = 2.0
    kernel_num = 5
    fix_sigma = 2326 # with 1000 columns

    # Balancing the samples to compute mmd using equal number of samples
    gtruth_X, pred_X = balance_source_target_sample_per_perturbation(gtruth_X, pred_X)

    mmd_dist = _compute_mmd(
        gtruth_X,
        pred_X,
        kernel_mul,
        kernel_num,
        fix_sigma,
        _gaussian_kernel
    )

    return mmd_dist


perturbations = gtruth.obs["gene"].cat.categories.tolist()
scores_mmd = []

for p in tqdm(perturbations):
    # Select true and predicted cells for perturbation p
    gt_mask = gtruth.obs["gene"] == p
    pr_mask = prediction.obs["gene"] == p

    # Extract their filtered expression matrices
    gtruth_X = to_array(gtruth[gt_mask, genes_for_metric])
    pred_X   = to_array(prediction[pr_mask, genes_for_metric])

    # Skip perturbations without matching predicted cells
    if gtruth_X.shape[0] == 0 or pred_X.shape[0] == 0:
        print(f"skipping {p}: missing samples: will not be accepted on the platform!")
        scores_mmd.append(9)
        continue

    # Compute MMD for this perturbation
    score = compute_metric_mmd(
        gtruth_X=gtruth_X,
        pred_X=pred_X,
    )

    scores_mmd.append(score)

print(f"MMD score: {np.mean(scores_mmd):.4f}")

  0%|          | 0/6 [00:00<?, ?it/s]

MMD score: 4.3851


#### Program-level metrics.

These metrics evaluate whether models capture meaningful biological outcomes.

#### L1-distance

There are four cell state proportions for each perturbation, i.e., pre-adipogenic, adipogenic, lipogenic, and other.
For each perturbation $p$, we have the ground truth cell-state proportion $S_p = [preadipo, adipo, lipo, other]$.

Let $S_p^R$ be the vector that has the proportion of the cell states $[preadipo, adipo, other]$. <br />
Let $S_p^L$ denote the condition probability of a cell being in the lipogenic state given the adipogenic state, i.e., $S_p^L = lipo/adipo$.

Then we define the program level loss as:

$
\displaystyle
L1(\hat{S}, S) = \frac{1}{|P|} \sum_{p \in P} 0.75 * | S_p^R - \hat{S}_p^R |_1 + 0.25 * | \hat{S}_p^L - S_p^L |
$

where $|.|_1$ is the L₁-distance, $|P|$ is the number of perturbations, and $\hat{S}_p$ is the predicted cell-state proportions.

---

*Lower is better!*

In [42]:
def compute_metric_l1_distance(
    true_state_proportion_df: pd.DataFrame,
    pred_state_proprotion_df: pd.DataFrame,
) -> float:
    # Going over all the genes that were perturbed in this set
    unique_perturb_genes = list(true_state_proportion_df["gene"].unique())

    all_l1_loss_list = []
    for gene in unique_perturb_genes:
        # Slicing the column with this gene
        true_gene_df = true_state_proportion_df[true_state_proportion_df["gene"] == gene]
        pred_gene_df = pred_state_proprotion_df[pred_state_proprotion_df["gene"] == gene]

        # print(gene, pred_gene_df.shape[0])
        assert true_gene_df.shape[0] == 1 and pred_gene_df.shape[0] == 1, f"Invalid prediction count for state gene={gene} count={pred_gene_df.shape[0]}!=1"

        # Getting the L1 loss for main  pre, adipo and other
        l1_three = (
            np.abs(true_gene_df.iloc[0]["pre_adipo"] - pred_gene_df.iloc[0]["pre_adipo"]) +
            np.abs(true_gene_df.iloc[0]["adipo"] - pred_gene_df.iloc[0]["adipo"]) +
            np.abs(true_gene_df.iloc[0]["other"] - pred_gene_df.iloc[0]["other"])
        )

        # Getting the L1 loss for lipo by adipo
        numerical_stab_term = 1e-20
        pred_lipo_adipo = pred_gene_df.iloc[0]["lipo"] / (pred_gene_df.iloc[0]["adipo"] + numerical_stab_term)
        true_lipo_adipo = true_gene_df.iloc[0]["lipo"] / (true_gene_df.iloc[0]["adipo"] + numerical_stab_term)
        l1_lipo_adipo = np.abs(true_lipo_adipo - pred_lipo_adipo)

        # Getting the average error
        average_l1 = 0.75 * l1_three + 0.25 * l1_lipo_adipo
        all_l1_loss_list.append(average_l1)

    # Getting the overall average over all the gene perturbation
    l1_loss = np.mean(all_l1_loss_list)
    return float(l1_loss)


l1_distance = compute_metric_l1_distance(
    gtruth_proportion,
    predicted_proportion,
)

print(f"L1-distance score: {l1_distance:.4f}")

L1-distance score: 0.2686


## Writing the report

The final step is to write the method description as specified [in the documentation](https://docs.crunchdao.com/competitions/competitions/broad-obesity/crunch-1#file-method-description.md).

You must:
1. Explain how your method works. *(5-10 sentences)*
2. Describe the reasoning behind your gene panel design. *(5-10 sentences)*
3. Specify the datasets and any other resources utilized. *(5-10 sentences)*

The limit is about one page.
<br />
<br />
<br />

---

<span style="font-size: 48px">👇👇👇</span> (double-click the markdown cell below)

---
file: Method description.md
---

<!-- Don't forget to change me -->

# Method Description

Explain how your method works. (5-10 sentences)

# Rationale

Describe the reasoning behind your gene panel design. (5-10 sentences)

# Data and Resources Used

Specify the datasets and any other resources utilized. (5-10 sentences)

<span style="font-size: 48px">👆👆👆</span>

---
<br />
<br />
<br />

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)